## 🔹 0. IMPORTS & CONFIGURATION

In [1]:
import numpy as np
import pandas as pd
from scipy.sparse import load_npz, issparse
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [12, 5]

## 1. CHARGER FEATURES & RÉSULTATS PRÉCÉDENTS

In [2]:
print("📥 Chargement des vecteurs, sentiments et topics...")

PATH_FEAT = "../../outputs/results/"
PATH_RAW  = "../../data/processed/mono-langue_clean.csv"

# 1. Vecteurs
X_tfidf = load_npz(f"{PATH_FEAT}mono_tfidf.npz")
X_sbert = np.load(f"{PATH_FEAT}mono_sbert.npy")

# 2. Dataset complet + résultats précédents
df = pd.read_csv(PATH_RAW)

# Assure-toi que ces colonnes existent (issues de 04 & 05)
# Si tu n'as pas de 'pred_sentiment', utilise 'sentiment' (ground truth)
df['pred_sentiment'] = df.get('pred_sentiment', df['sentiment'])
df['lda_topic']      = df.get('lda_topic', df.get('bert_topic', 0))
df['bert_topic']     = df.get('bert_topic', df.get('lda_topic', 0))

# Mapping sentiment -> score numerique (pour ponderation)
sentiment_to_score = {'positive': 1.0, 'neutral': 0.5, 'negative': 0.0}
df['sent_score'] = df['pred_sentiment'].map(sentiment_to_score).fillna(0.5)

print(f"✅ {len(df)} reviews chargees")
print(f"✅ Colonnes disponibles : {list(df.columns)}")

📥 Chargement des vecteurs, sentiments et topics...
✅ 500 reviews chargees
✅ Colonnes disponibles : ['review_id', 'text', 'rating', 'sentiment', 'product_category', 'text_clean_classique', 'text_clean_light', 'pred_sentiment', 'lda_topic', 'bert_topic', 'sent_score']


### 🔹 2. MOTEUR DE RECOMMANDATION (SCORE HYBRIDE)

In [3]:
def compute_recommendation_scores(query_idx, X, strategy='hybrid', 
                                  w_sim=0.5, w_sent=0.3, w_topic=0.2, 
                                  topic_col='bert_topic', boost_positive=True):
    """
    Calcule les scores de recommandation selon la strategie choisie.
    Retourne : indices tries, scores detailles
    """
    # 1. Similarite textuelle (base commune)
    query_vec = X[query_idx]
    if not issparse(query_vec):
        query_vec = np.atleast_2d(query_vec)
    sim_scores = cosine_similarity(query_vec, X).flatten()
    sim_scores[query_idx] = -1  # Exclure la review elle-meme

    # Normalisation 0-1
    sim_norm = np.clip(sim_scores, 0, 1)

    # 2. Score Sentiment
    if strategy in ['sentiment', 'hybrid']:
        q_sent = df.iloc[query_idx]['pred_sentiment']
        # Match de sentiment
        sent_match = (df['pred_sentiment'] == q_sent).astype(float).to_numpy()
        
        # Boost/Penalite globale (favoriser positif, penaliser negatif)
        sent_boost = np.zeros(len(df))
        if boost_positive:
            sent_boost[df['pred_sentiment'] == 'positive'] = 0.2
            sent_boost[df['pred_sentiment'] == 'negative'] = -0.2
            
        sent_final = sent_match + sent_boost
    else:
        sent_final = np.zeros(len(df))

    # 3. Score Topic
    if strategy in ['topic', 'hybrid']:
        q_topic = df.iloc[query_idx][topic_col]
        topic_match = (df[topic_col] == q_topic).astype(float).to_numpy()
    else:
        topic_match = np.zeros(len(df))

    # 4. Combinaison ponderee
    final_scores = w_sim * sim_norm + w_sent * sent_final + w_topic * topic_match
    final_scores[query_idx] = -1

    # Top indices
    top_indices = np.argsort(final_scores)[::-1]

    return top_indices, {
        'sim': sim_scores,
        'sent': sent_final,
        'topic': topic_match,
        'final': final_scores
    }

## 🔹 3. BASELINE : TF-IDF & EMBEDDINGS

In [4]:
def run_baseline(X, name, query_idx, top_k=5):
    print(f"\n🔵 BASELINE : {name} (Cosine Similarity)")
    idx, scores = compute_recommendation_scores(query_idx, X, strategy='baseline')
    top_k_idx = idx[:top_k]
    
    for i, rec_idx in enumerate(top_k_idx, 1):
        print(f"{i}. [{scores['sim'][rec_idx]:.3f}] {df.iloc[rec_idx]['text_clean_light'][:70]}...")
    return top_k_idx

# Exemple : prenons une review au hasard ou specifique
query_idx = 42  # <- Change selon tes besoins
print(f"📖 Review cible ({query_idx}) : \"{df.iloc[query_idx]['text_clean_light'][:80]}...\"")

# Lancer les deux baselines
baseline_tfidf = run_baseline(X_tfidf, "TF-IDF", query_idx)
baseline_sbert = run_baseline(X_sbert, "SBERT Embeddings", query_idx)

📖 Review cible (42) : "it works as expected..."

🔵 BASELINE : TF-IDF (Cosine Similarity)
1. [1.000] it works as expected...
2. [1.000] it works as expected...
3. [1.000] it works as expected...
4. [1.000] it works as expected...
5. [1.000] it works as expected...

🔵 BASELINE : SBERT Embeddings (Cosine Similarity)
1. [1.000] it works as expected...
2. [1.000] it works as expected...
3. [1.000] it works as expected...
4. [1.000] it works as expected...
5. [1.000] it works as expected...


## 🔹 4. SYSTÈMES ENRICHIS (Sentiment, Topic, Hybride)

In [5]:
def run_enhanced(X, strategy, query_idx, top_k=5, **kwargs):
    print(f"\n STRATEGIE : {strategy.upper()}")
    idx, scores = compute_recommendation_scores(query_idx, X, strategy=strategy, **kwargs)
    top_k_idx = idx[:top_k]
    
    for i, rec_idx in enumerate(top_k_idx, 1):
        sim = scores['sim'][rec_idx]
        sent = scores['sent'][rec_idx]
        topic = scores['topic'][rec_idx]
        print(f"{i}. [Sim:{sim:.2f} | Sent:{sent:.2f} | Topic:{topic:.1f}] "
              f"{df.iloc[rec_idx]['text_clean_light'][:60]}...")
    return top_k_idx

# Lancer les 3 strategies enrichies (sur SBERT pour meilleure similarite)
sent_rec   = run_enhanced(X_sbert, 'sentiment', query_idx, w_sim=0.6, w_sent=0.4)
topic_rec  = run_enhanced(X_sbert, 'topic', query_idx, w_sim=0.6, w_topic=0.4)
hybrid_rec = run_enhanced(X_sbert, 'hybrid', query_idx, w_sim=0.5, w_sent=0.3, w_topic=0.2)


 STRATEGIE : SENTIMENT
1. [Sim:1.00 | Sent:1.00 | Topic:0.0] it works as expected...
2. [Sim:1.00 | Sent:1.00 | Topic:0.0] it works as expected...
3. [Sim:1.00 | Sent:1.00 | Topic:0.0] it works as expected...
4. [Sim:1.00 | Sent:1.00 | Topic:0.0] it works as expected...
5. [Sim:1.00 | Sent:1.00 | Topic:0.0] it works as expected...

 STRATEGIE : TOPIC
1. [Sim:1.00 | Sent:0.00 | Topic:1.0] it works as expected...
2. [Sim:1.00 | Sent:0.00 | Topic:1.0] it works as expected...
3. [Sim:1.00 | Sent:0.00 | Topic:1.0] it works as expected...
4. [Sim:1.00 | Sent:0.00 | Topic:1.0] it works as expected...
5. [Sim:1.00 | Sent:0.00 | Topic:1.0] it works as expected...

 STRATEGIE : HYBRID
1. [Sim:1.00 | Sent:1.00 | Topic:1.0] it works as expected...
2. [Sim:1.00 | Sent:1.00 | Topic:1.0] it works as expected...
3. [Sim:1.00 | Sent:1.00 | Topic:1.0] it works as expected...
4. [Sim:1.00 | Sent:1.00 | Topic:1.0] it works as expected...
5. [Sim:1.00 | Sent:1.00 | Topic:1.0] it works as expected...


## 🔹 5. EXPLICATIONS & INTERPRÉTABILITÉ

In [6]:
def explain_recommendation(query_idx, rec_idx, scores):
    q_sent = df.iloc[query_idx]['pred_sentiment']
    r_sent = df.iloc[rec_idx]['pred_sentiment']
    q_topic = df.iloc[query_idx]['bert_topic']
    r_topic = df.iloc[rec_idx]['bert_topic']
    
    reasons = []
    if scores['sim'][rec_idx] > 0.6: reasons.append("similarite textuelle forte")
    if scores['sent'][rec_idx] > 0: reasons.append("sentiment aligne/positif")
    if scores['topic'][rec_idx] == 1: reasons.append("meme topic thematique")
    if r_sent == 'positive': reasons.append("avis positif (booste)")
    elif r_sent == 'negative': reasons.append("avis negatif (penalise)")
    
    print(f"✅ Recommended because: {', '.join(reasons)}")

print("\n🔍 EXPLICATION DETAILLEE (Top 1 Hybride) :")
explain_recommendation(query_idx, hybrid_rec[0], compute_recommendation_scores(query_idx, X_sbert, 'hybrid')[1])


🔍 EXPLICATION DETAILLEE (Top 1 Hybride) :
✅ Recommended because: similarite textuelle forte, sentiment aligne/positif, meme topic thematique


## 🔹 6. RECOMMANDATION PRODUITS / CATÉGORIES

In [7]:
print("\n RECOMMANDATION DE PRODUITS / CATEGORIES")

# Agregation par categorie
cat_scores = df.groupby('product_category').agg({
    'pred_sentiment': lambda x: (x == 'positive').mean(),  # % positif
    'bert_topic': lambda x: x.mode()[0] if not x.mode().empty else None
}).rename(columns={'pred_sentiment': 'positive_ratio', 'bert_topic': 'dominant_topic'})

# Recommander la categorie la plus proche du topic/query
query_topic = df.iloc[query_idx]['bert_topic']
query_sent = df.iloc[query_idx]['pred_sentiment']

# Score categorie = similarite topic + ratio positif
cat_scores['rec_score'] = 0.0
cat_scores.loc[cat_scores['dominant_topic'] == query_topic, 'rec_score'] += 0.6
cat_scores['rec_score'] += cat_scores['positive_ratio'] * 0.4

top_cats = cat_scores.sort_values('rec_score', ascending=False).head(3)
print("\n Categories recommandees :")
print(top_cats[['positive_ratio', 'dominant_topic', 'rec_score']].to_markdown())


 RECOMMANDATION DE PRODUITS / CATEGORIES

 Categories recommandees :
| product_category   |   positive_ratio |   dominant_topic |   rec_score |
|:-------------------|-----------------:|-----------------:|------------:|
| electronics        |         0.360947 |                0 |    0.744379 |
| home               |         0.308989 |                0 |    0.723596 |
| fashion            |         0.248366 |                0 |    0.699346 |


## 🔹 7. ÉVALUATION & COMPARAISON DES SYSTÈMES

In [8]:
print("\n COMPARAISON DES SYSTÈMES DE RECOMMANDATION")

comparison_data = []
strategies = ['baseline', 'sentiment', 'topic', 'hybrid']

for strat in strategies:
    idx, scores = compute_recommendation_scores(query_idx, X_sbert, strategy=strat)
    top5 = idx[:5]
    
    # Métriques qualitatives rapides
    avg_sim = scores['sim'][top5].mean()
    pos_ratio = (df.loc[top5, 'pred_sentiment'] == 'positive').mean()
    topic_match = (scores['topic'][top5] == 1).sum()
    
    comparison_data.append({
        'Système': strat.capitalize(),
        'Similarité Moy.': f"{avg_sim:.3f}",
        '% Positifs': f"{pos_ratio:.1%}",
        'Topic Matchs': topic_match,
        'Utilise': 'Similarité textuelle' if strat=='baseline' else 
                   'Similarité + Sentiment' if strat=='sentiment' else
                   'Similarité + Topics' if strat=='topic' else
                   'Similarité + Sentiment + Topics'
    })

df_comp = pd.DataFrame(comparison_data)
print(df_comp.to_markdown(index=False))


 COMPARAISON DES SYSTÈMES DE RECOMMANDATION
| Système   |   Similarité Moy. | % Positifs   |   Topic Matchs | Utilise                         |
|:----------|------------------:|:-------------|---------------:|:--------------------------------|
| Baseline  |                 1 | 0.0%         |              0 | Similarité textuelle            |
| Sentiment |                 1 | 0.0%         |              0 | Similarité + Sentiment          |
| Topic     |                 1 | 0.0%         |              5 | Similarité + Topics             |
| Hybrid    |                 1 | 0.0%         |              5 | Similarité + Sentiment + Topics |
